# Feature Stores & Data Infrastructure

**Chapter 8: AI Infrastructure**

This notebook demonstrates end-to-end feature store setup using **Feast** (open-source, local) for a recommendation system.

**Running Example:**
- System: Document type recommender for InferenceBase API
- Features: User context (last 10 uploads, avg confidence) + document metadata (page count, language)
- Stores: SQLite offline store (training), Redis online store (serving)

**Tools:**
- **Feast**: Feature store framework
- **Redis**: Online store (low-latency serving)
- **Parquet**: Offline store (historical training data)

**Objectives:**
1. Define feature views (user + document features)
2. Materialize features (offline → online store)
3. Fetch online features (serving, <10ms)
4. Fetch historical features (training, point-in-time correct)
5. Measure latency (Redis vs direct DB)
6. Version features (reproduce training data)

---

## Cell 1: Setup Feast + Redis

Install Feast with Redis support and initialize feature repository.

**What we're setting up:**
- Feast CLI and Python SDK
- Redis (online store for <10ms feature lookups)
- Local Parquet offline store (training data)

**Architecture:**
```
Data Sources (Parquet)
 ↓
Feature Definitions (Python)
 ↓
Offline Store (Parquet) ← Training
Online Store (Redis) ← Serving
```

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
# 2. Compute `repo_path` using `Path()`
# 3. Call `chdir()` to produce the result
# 4. Call `Redis()` to produce the result
# 5. Call `rglob()` to produce the result
#
# Hint:
#    repo_path = Path(???)
#    r = redis.Redis(???)

## Cell 2: Generate Synthetic User Activity Data

Create training data simulating InferenceBase user behavior:
- User uploads (document type, confidence score, page count)
- Historical events with timestamps (for point-in-time joins)

**Schema:**
- `user_id`: int (1-1000)
- `event_timestamp`: datetime
- `doc_type`: str (invoice, contract, receipt, tax_form)
- `confidence_score`: float (0.6-1.0)
- `page_count`: int (1-50)
- `has_tables`: bool

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
# 2. Call `seed()` to produce the result
# 3. Compute `n_events` using `events()`
# 4. Compute `doc_types`
# 5. Compute `user_events` using `randint()`
# 6. Compute `user_events['created_at']` using `created_at()`
# 7. Compute `data_dir` using `Parquet()`
# 8. Call `nunique()` to produce the result
#
# Hint:
#    data_dir = Path(???)
#    start_date = datetime.now(???)
#    user_events = pd.DataFrame(???)

## Cell 3: Define Feature Views (User + Document Features)

Create feature definitions that Feast will use to generate both training and serving features.

**Feature views:**
1. **user_activity_features** — User behavior over last 30 days
 - `avg_confidence_score`: Mean confidence of user's documents
 - `total_pages_processed`: Sum of all pages processed
 - `upload_count`: Number of documents uploaded
 - `most_common_doc_type`: User's preferred document type

2. **user_doc_type_affinity** — Per-user preference for each doc type
 - `invoice_ratio`: % of user's uploads that are invoices
 - `contract_ratio`: % contracts
 - `receipt_ratio`: % receipts
 - `tax_form_ratio`: % tax forms

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `features_code`
# 2. Compute `user` using `entity()`
# 3. Compute `user_uploads_source` using `source()`
# 4. Compute `user_activity_features` using `Field()`
# 5. Compute `user_doc_affinity` using `Field()`
# 6. Call `write()` to produce the result
# 7. Call `user_activity_features()` to produce the result
#
# Hint:
#    user = Entity(name=???, join_keys=???)
#    user_uploads_source = FileSource(name=???, path=???)

## Cell 4: Configure Feature Store (Redis + Parquet)

Update `feature_store.yaml` to use:
- **Online store**: Redis (localhost:6379)
- **Offline store**: Local Parquet files
- **Registry**: SQLite (feature metadata)

Then apply feature definitions to the registry.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `config`
# 2. Call `write()` to produce the result
# 3. Call `Redis()` to produce the result
# 4. Compute `result` using `run()`
# 5. Process data
# 6. Compute `fs` using `FeatureStore()`
# 7. Call `list_feature_views()` to produce the result
#
# Hint:
#    fs = FeatureStore(repo_path=???)
#    result = subprocess.run(???)

## Cell 5: Compute Features + Materialize to Online Store

**Materialization** = compute features from raw data → write to online store

**Process:**
1. Read `user_uploads.parquet`
2. Aggregate per user:
 - `AVG(confidence_score)`
 - `SUM(page_count)`
 - `COUNT(*)`
 - `doc_type` ratios
3. Write to Redis:
 - Key: `user:12345:avg_confidence_score`
 - Value: `0.87`

**Note:** In production, this runs on a schedule (hourly/daily via Airflow).

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
# 2. Compute `user_uploads` using `read_parquet()`
# 3. Aggregate data into `user_activity` -- use `groupby()`
# 4. Compute `user_activity['created_at']` using `now()`
# 5. Aggregate data into `doc_affinity` -- use `groupby()`
# 6. Aggregate data into `user_doc_affinity_features['event_timestamp']` -- use `groupby()`
# 7. Call `to_parquet()` to produce the result
# 8. Call `head()` to produce the result
# 9. Compute `features_code_updated`
# 10. Compute `user` using `Entity()`
# 11. Compute `user_activity_source` using `FileSource()`
# 12. Compute `user_doc_affinity_source` using `FileSource()`
# 13. Compute `user_activity_features` using `Field()`
# 14. Compute `user_doc_affinity` using `Field()`
# 15. Call `write()` to produce the result
# 16. Call `run()` to produce the result
# 17. Compute `end_date` using `store()`
# 18. Compute `result` using `isoformat()`
# 19. Process data
#
# Hint:
#    user = Entity(name=???, join_keys=???)
#    user_activity_source = FileSource(name=???, path=???)
#    user_uploads = pd.read_parquet(???)
#    user_activity = user_uploads.groupby(???)

## Cell 6: Fetch Online Features (Real-Time Serving)

Simulate production inference: fetch features for a single user in <10ms.

**Use case:** User 42 uploads a new document → API needs features to predict document type.

**Latency target:** <10ms (p95)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
# 2. Compute `fs` using `FeatureStore()`
# 3. Compute `entity_rows`
# 4. Compute `features_to_fetch`
# 5. Compute `latencies` using `perf_counter()`
# 6. Call `Serving()` to produce the result
# 7. Call `percentile()` to produce the result
#
# Hint:
#    fs = FeatureStore(repo_path=???)
#    start = time.perf_counter(???)
#    online_features = fs.get_online_features(???)

## Cell 7: Fetch Historical Features (Training Data)

**Point-in-time correct joins** — fetch features as they existed at specific timestamps.

**Why this matters:**
- Prevents data leakage (model can't see future data during training)
- Reproduces exact training environment

**Example:** For a training sample at timestamp T, fetch features computed using only data BEFORE T.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Compute `training_timestamps` using `now()`
# 2. Compute `entity_df` using `DataFrame()`
# 3. Call `nunique()` to produce the result
# 4. Compute `start` using `perf_counter()`
# 5. Call `head()` to produce the result
# 6. Compute `null_counts` using `nulls()`
# 7. Compute `sample_row`
#
# Hint:
#    training_timestamps = pd.date_range(???)
#    start = datetime.now(???)
#    end = datetime.now(???)
#    entity_df = pd.DataFrame(???)

## Cell 8: Measure Latency — Feature Store vs Direct DB

Compare latency of:
1. **Feature store (Redis)**: Precomputed features, <10ms lookup
2. **Direct DB (simulated)**: On-the-fly aggregation, ~200-400ms

**Business impact:**
- Before: 380ms feature queries → 2.8s p95 total latency (violated SLA)
- After: 8ms feature queries → 1.4s p95 total latency (30% better than target)

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
# 2. Compute `feast_latencies` using `latency()`
# 3. Aggregate data into `db_latencies` -- use `perf_counter()`
# 4. Call `percentile()` to produce the result
# 5. Compute `p95_improvement` using `percentile()`
# 6. Plot results -- call `subplots()`
# 7. Call `boxplot()` to produce the result
# 8. Call `hist()` to produce the result
# 9. Plot results -- call `tight_layout()`
# 10. Process data
#
# Hint:
#    start = time.perf_counter(???)

## Cell 9: Point-in-Time Join Validation (Prevent Data Leakage)

**Critical test:** Verify that historical features don't use future data.

**Test:**
1. Fetch features for timestamp T
2. Verify features only use data with `event_timestamp < T`
3. Confirm no "future peeking" that would inflate training accuracy

**Why this matters:** Data leakage is a silent killer — model looks great in training, fails in production.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
# 2. Compute `test_user_id` using `now()`
# 3. Process data
# 4. Compute `events_before`
# 5. Call `mean()` to produce the result
# 6. Compute `entity_df_test` using `Feast()`
# 7. Compute `feast_features` using `get_historical_features()`
# 8. Call `features()` to produce the result
# 9. Call `mean()` to produce the result
# 10. Call `correct()` to produce the result
# 11. Compute `events_after`
# 12. Process data
#
# Hint:
#    test_timestamp = datetime.now(???)
#    entity_df_test = pd.DataFrame(???)
#    feast_features = fs.get_historical_features(???)

## Cell 10: Feature Versioning + Reproducibility

**Problem:** Feature definitions change over time. How do you reproduce training data from 3 months ago?

**Solution:** Git-based feature versioning.

**Workflow:**
1. Train model → tag Git commit with feature definitions
2. Log commit SHA in MLflow (from Ch.9)
3. Retrain later → checkout old Git tag → reproduce exact features

**Demo:** Show how feature definitions are version-controlled.

In [ ]:
# TODO: Implement this cell
#
# Steps:
# 1. Process data
# 2. Call `run()` to produce the result
# 3. Compute `feature_snapshot` using `now()`
# 4. Compute `snapshot_path` using `Path()`
# 5. Call `dumps()` to produce the result
# 6. Call `commit()` to produce the result
# 7. Call `list_feature_views()` to produce the result
# 8. Call `mo()` to produce the result
#
# Hint:
#    snapshot_path = Path(???)
#    result = subprocess.run(???)
#    current_commit = result.stdout.strip(???)